# Modelo A5 — Híbrido con popularidad protegida

## Hipótesis

El análisis del Modelo A2 mostró que la popularidad global y la señal de perfil contienen información complementaria.

En el conjunto de test:

- El baseline de popularidad obtuvo 37 aciertos.
- A2 obtuvo 36 aciertos.
- 26 aciertos fueron compartidos.
- 10 fueron exclusivos de A2.
- 11 fueron exclusivos del baseline.

Además, A2 no reproduce simplemente el ranking de popularidad. En promedio, 6,37 de sus 10 recomendaciones pertenecen al Top-10 global, mientras que el resto cambia según la señal personalizada.

Esto sugiere que la personalización permite recuperar productos que el baseline no encuentra, pero también puede desplazar productos muy populares que resultaban relevantes.

## Propuesta

A5 busca conservar ambas señales mediante un ranking híbrido con popularidad protegida.

El Top-10 final se construirá reservando una cantidad limitada de posiciones para productos de alta popularidad global y completando las posiciones restantes con el ranking personalizado.

La cantidad de posiciones protegidas no se seleccionará utilizando el conjunto de test.

Se evaluarán distintas configuraciones exclusivamente sobre validación y se elegirá la mejor según HR@10 y NDCG@10.

Una vez seleccionada la configuración, se realizará una única evaluación sobre test.

## Objetivo

Evaluar si es posible conservar parte de los aciertos aportados por la popularidad sin perder la capacidad de A2 de introducir productos relevantes para perfiles específicos.

In [2]:
from pathlib import Path
import sys
from collections import Counter

import numpy as np
import pandas as pd

from sklearn.feature_extraction import DictVectorizer
from sklearn.metrics.pairwise import cosine_similarity

from IPython.display import display


# Buscar la raíz del proyecto
PROJECT_ROOT = next(
    (
        p for p in [Path.cwd(), *Path.cwd().parents]
        if (p / "data" / "processed").is_dir()
    ),
    None
)

if PROJECT_ROOT is None:
    raise FileNotFoundError(
        "No se encontró la carpeta data/processed"
    )


# Buscar sephora_common.py
helper_dir = PROJECT_ROOT / "notebooks" / "Etapa_02"

if not (helper_dir / "sephora_common.py").exists():
    raise FileNotFoundError(
        "No se encontró sephora_common.py en Etapa_02"
    )

sys.path.insert(0, str(helper_dir))

import sephora_common as sc
from sephora_common import PROFILE_COLS, rank_scores


# Mantenemos exactamente el mismo protocolo
N_VALIDACION = 500
N_TEST = 500
SEMILLA = 42

ctx = sc.load_context(
    N_VALIDACION,
    N_TEST,
    SEMILLA
)


print("Protocolo:", ctx.protocol_id[:12])
print(
    "Usuarios entrenamiento:",
    ctx.train.author_id.nunique()
)
print(
    "Productos del catálogo:",
    ctx.catalog.product_id.nunique()
)

Protocolo: 99fc64b0ea63
Usuarios entrenamiento: 502216
Productos del catálogo: 2420


## 3. Modelo A5 — Popularidad protegida

A5 conserva la lógica de A2 para calcular la señal personalizada mediante usuarios con perfiles similares.

La diferencia se encuentra en la construcción del Top-10 final.

En lugar de combinar uniformemente popularidad y perfil para todas las posiciones, se reserva una cantidad limitada de posiciones para los productos con mayor popularidad global. Las posiciones restantes se completan utilizando el ranking personalizado.

De esta forma se busca conservar parte de la fortaleza del baseline sin eliminar la capacidad del modelo de recomendar productos diferentes según el perfil del usuario.

La cantidad de posiciones protegidas será seleccionada exclusivamente utilizando el conjunto de validación.

In [3]:
class ProfilePopularityProtected:
    """
    Recomendador híbrido con popularidad protegida.

    Una parte del Top-10 se reserva para productos
    globalmente populares.

    Las posiciones restantes se completan utilizando
    la señal personalizada de usuarios con perfiles similares.
    """

    def __init__(self, ctx):

        # Usuarios que pertenecen al conjunto de entrenamiento
        self.profiles = ctx.profiles.loc[
            sorted(ctx.train.author_id.unique())
        ]

        # Convertimos las características del perfil
        # a variables numéricas
        self.encoder = DictVectorizer(sparse=True)

        records = self.profiles.to_dict("records")

        self.encoder.fit(
            records + [{c: "Unknown" for c in PROFILE_COLS}]
        )

        # Valores permitidos para cada característica
        self.allowed = {
            c: set(self.profiles[c]) | {"Unknown"}
            for c in PROFILE_COLS
        }

        # Agrupamos usuarios con el mismo perfil
        self.groups = {}

        for uid, row in self.profiles.iterrows():

            perfil = tuple(
                row[c] for c in PROFILE_COLS
            )

            self.groups.setdefault(
                perfil,
                []
            ).append(uid)

        # Guardamos los perfiles distintos
        self.signatures = list(
            self.groups
        )

        # Transformamos los perfiles
        self.X = self.encoder.transform([
            dict(zip(PROFILE_COLS, perfil))
            for perfil in self.signatures
        ])

        # Productos positivos de cada usuario
        self.positives = (
            ctx.train[ctx.train.positive]
            .groupby("author_id")
            .product_id
            .apply(list)
            .to_dict()
        )

        # Popularidad global calculada solamente con train
        popularidad = (
            ctx.train[ctx.train.positive]
            .groupby("product_id")
            .size()
            .reset_index(name="popularity")
            .sort_values(
                ["popularity", "product_id"],
                ascending=[False, True]
            )
        )

        # Ranking global de popularidad
        self.popular_products = (
            popularidad["product_id"]
            .tolist()
        )

        self.n_train_users = len(
            self.profiles
        )


    def neighbors(self, profile, k):

        if not 1 <= k <= self.n_train_users:
            raise ValueError(
                f"K={k} inválido para "
                f"{self.n_train_users} usuarios de entrenamiento"
            )

        # Armamos el perfil del usuario
        key = tuple(
            profile.get(c, "Unknown")
            if profile.get(c, "Unknown") in self.allowed[c]
            else "Unknown"
            for c in PROFILE_COLS
        )

        query = self.encoder.transform([
            dict(zip(PROFILE_COLS, key))
        ])

        # Similitud entre perfiles
        similarities = cosine_similarity(
            query,
            self.X
        ).ravel()

        vecinos = []

        orden = np.argsort(
            -similarities
        )

        for indice in orden:

            similitud = similarities[indice]

            usuarios_grupo = self.groups[
                self.signatures[indice]
            ]

            for uid in usuarios_grupo:

                vecinos.append(
                    (uid, similitud)
                )

                if len(vecinos) == k:
                    return vecinos

        return vecinos


    def recommend(
        self,
        profile,
        config,
        top_n=10,
        exclude=()
    ):

        k = int(
            config["k"]
        )

        n_protegidos = int(
            config["n_protegidos"]
        )

        # ---------------------------------------------
        # 1. Ranking personalizado
        # ---------------------------------------------

        profile_scores = Counter()

        vecinos = self.neighbors(
            profile,
            k
        )

        for uid, similitud in vecinos:

            productos_usuario = self.positives.get(
                uid,
                []
            )

            for product_id in productos_usuario:

                profile_scores[product_id] += similitud

        # Ordenamos los productos según la señal de perfil
        ranking_perfil = sorted(
            profile_scores.items(),
            key=lambda x: (-x[1], str(x[0]))
        )

        # Guardamos solamente los product_id
        ranking_perfil = [
            product_id
            for product_id, score in ranking_perfil
            if product_id not in set(exclude)
        ]

        # ---------------------------------------------
        # 2. Productos de popularidad protegida
        # ---------------------------------------------

        excluidos = set(exclude)

        protegidos = []

        for product_id in self.popular_products:

            if product_id in excluidos:
                continue

            protegidos.append(
                product_id
            )

            if len(protegidos) == n_protegidos:
                break

        # ---------------------------------------------
        # 3. Completamos con el ranking personalizado
        # ---------------------------------------------

        seleccionados = list(
            protegidos
        )

        for product_id in ranking_perfil:

            if product_id not in seleccionados:

                seleccionados.append(
                    product_id
                )

            if len(seleccionados) == top_n:
                break

        # Si faltaran productos, completamos con popularidad
        if len(seleccionados) < top_n:

            for product_id in self.popular_products:

                if (
                    product_id not in seleccionados
                    and product_id not in excluidos
                ):

                    seleccionados.append(
                        product_id
                    )

                if len(seleccionados) == top_n:
                    break

       # ---------------------------------------------
        # 4. Devolvemos el Top-10 seleccionado
        # ---------------------------------------------

        return seleccionados[:top_n]

In [4]:
# Volvemos a crear el modelo con la clase corregida
model = ProfilePopularityProtected(ctx)

# Tomamos el primer caso de validación correctamente
caso_prueba = ctx.validation.iloc[0]

author_id = caso_prueba["author_id"]

# Buscamos el perfil del usuario
perfil_prueba = ctx.profiles.loc[
    author_id
].to_dict()

# Generamos un Top-10 de prueba
recomendaciones_prueba = model.recommend(
    perfil_prueba,
    {
        "k": 500,
        "n_protegidos": 3
    },
    top_n=10
)

print("Recomendaciones:")
print(recomendaciones_prueba)

print()
print("Cantidad:", len(recomendaciones_prueba))

print(
    "Sin duplicados:",
    len(recomendaciones_prueba)
    == len(set(recomendaciones_prueba))
)

print(
    "Todos pertenecen al catálogo:",
    set(recomendaciones_prueba).issubset(
        set(ctx.catalog.product_id)
    )
)

Recomendaciones:
['P420652', 'P7880', 'P269122', 'P430337', 'P427419', 'P411540', 'P471546', 'P248407', 'P394624', 'P394639']

Cantidad: 10
Sin duplicados: True
Todos pertenecen al catálogo: True


In [5]:
# Crear el modelo A5
model = ProfilePopularityProtected(ctx)

# Configuraciones que vamos a comparar SOLO en validación
configs = []

for k in [100, 250, 500]:
    for n_protegidos in [1, 2, 3, 4]:
        
        configs.append({
            "k": k,
            "n_protegidos": n_protegidos
        })

print("Cantidad de configuraciones:", len(configs))

display(
    pd.DataFrame(configs)
)

Cantidad de configuraciones: 12


,k,n_protegidos
0,100,1
1,100,2
2,100,3
3,100,4
4,250,1
5,250,2
6,250,3
7,250,4
8,500,1
9,500,2


In [6]:
# Evaluamos las 12 configuraciones SOLO en validación

resultados_validacion = []

for config in configs:

    metricas, _ = sc.evaluate(
        model,
        ctx,
        ctx.validation,
        config,
        verbose=False
    )

    resultados_validacion.append({
        "k": config["k"],
        "n_protegidos": config["n_protegidos"],
        "HR@10": metricas["hit_rate_at_10"],
        "NDCG@10": metricas["ndcg_at_10"]
    })


# Armamos la tabla
tabla_validacion = pd.DataFrame(
    resultados_validacion
)

# Ordenamos primero por HR y luego por NDCG
tabla_validacion = tabla_validacion.sort_values(
    ["HR@10", "NDCG@10"],
    ascending=False
).reset_index(drop=True)

display(tabla_validacion)

,k,n_protegidos,HR@10,NDCG@10
0,250,2,0.070,0.038361
1,250,1,0.068,0.039818
2,500,1,0.068,0.038544
3,250,4,0.068,0.036519
4,500,2,0.066,0.036443
5,500,3,0.066,0.036131
6,250,3,0.064,0.035660
7,500,4,0.062,0.034757
8,100,4,0.056,0.032652
9,100,1,0.054,0.032932


In [7]:
# Seleccionar automáticamente la mejor configuración
# según los resultados de validación
mejor_fila = tabla_validacion.iloc[0]

mejor_config = {
    "k": int(mejor_fila["k"]),
    "n_protegidos": int(mejor_fila["n_protegidos"])
}

print("Mejor configuración según validación:", mejor_config)

Mejor configuración según validación: {'k': 250, 'n_protegidos': 2}


In [8]:
# Seleccionar la mejor configuración según VALIDACIÓN
# Primero se prioriza HR@10 y, en caso de empate, NDCG@10

mejor_fila = tabla_validacion.iloc[0]

mejor_config = {
    "k": int(mejor_fila["k"]),
    "n_protegidos": int(mejor_fila["n_protegidos"])
}

print(
    "Mejor configuración seleccionada en validación:",
    mejor_config
)

# Evaluación final de A5 en TEST
metricas_test_a5, detalle_test_a5 = sc.evaluate(
    model,
    ctx,
    ctx.test,
    mejor_config,
    verbose=False
)

# Evaluar el mismo baseline global utilizado en A1
baseline_test, _ = sc.evaluate(
    sc.PopularityBaseline(ctx),
    ctx,
    ctx.test,
    {},
    verbose=False
)

print("Comparación A5 vs. baseline global:")

display(
    pd.DataFrame(
        [metricas_test_a5, baseline_test],
        index=["A5", "Baseline global"]
    )
)

# Mostrar métricas finales
display(
    pd.DataFrame(
        [metricas_test_a5],
        index=["A5"]
    )
)

Mejor configuración seleccionada en validación: {'k': 250, 'n_protegidos': 2}
Comparación A5 vs. baseline global:


,users,hits,hit_rate_at_10,ndcg_at_10,user_coverage,full_top10_rate,catalog_coverage,mean_recommendations,mean_query_seconds
A5,500,29,0.058,0.026809,1.0,1.0,0.095455,10.0,0.004271
Baseline global,500,37,0.074,0.033161,1.0,1.0,0.004132,10.0,0.006653


,users,hits,hit_rate_at_10,ndcg_at_10,user_coverage,full_top10_rate,catalog_coverage,mean_recommendations,mean_query_seconds
A5,500,29,0.058,0.026809,1.0,1.0,0.095455,10.0,0.004271


In [9]:
# Generar recomendaciones de ejemplo con la configuración ganadora
caso_ejemplo = ctx.validation.iloc[0]
author_id_ejemplo = caso_ejemplo["author_id"]

perfil_ejemplo = ctx.profiles.loc[author_id_ejemplo].to_dict()

recomendaciones_finales = model.recommend(
    perfil_ejemplo,
    mejor_config,
    top_n=10
)

# Preparar tabla para guardar
top10_a5 = pd.DataFrame({
    "author_id": [author_id_ejemplo] * len(recomendaciones_finales),
    "product_id": recomendaciones_finales
})

# Guardar métricas, configuración y recomendaciones
archivo = sc.save_result(
    ctx,
    "E2_A5_popularidad_protegida",
    mejor_config,
    tabla_validacion,
    metricas_test_a5,
    detalle_test_a5,
    baseline_test,
    top10_a5,
    {"algorithm": "ProfilePopularityProtected"}
)

print("Resultado guardado:", archivo)

Resultado guardado: c:\Users\Administrator\Documents\DATA SCIENCE\PF\sephora-recommender\artifacts\Etapa_02\E2_A5_popularidad_protegida.json
